# Qwen PDF QA SFT (Regression-Hardened)

This notebook is designed to fine-tune Qwen on BiB PDF papers **without introducing transcript-style behavior drift** (for example: role-prefix leakage, prompt echo, and repetition loops).

## Why this notebook is different
- Uses **task-aligned QA + abstention** examples instead of generic sentence-restatement only.
- Uses **document-level split** to avoid train/eval leakage across chunks from the same PDF.
- Uses **assistant-only loss masking** so the model is trained to generate only the assistant answer text.
- Adds **format guardrails** and a post-training **behavior gate** to catch role-prefix and repetition regressions before deployment.

## Output artifacts
1. LoRA adapter
2. Merged model
3. Optional 4-bit merged model

In [ ]:
import os

from pathlib import Path

from google.colab import drive



drive.mount('/content/drive')



EVAL_ROOT = Path('/content/drive/MyDrive/eval')

EVAL_ROOT.mkdir(parents=True, exist_ok=True)



MODEL_TRAINING_DIR = EVAL_ROOT / 'model_training'

MODEL_TRAINING_DIR.mkdir(parents=True, exist_ok=True)



# Write marker first to confirm Drive write access for checkpoints/artifacts.

marker = EVAL_ROOT / '_write_ok.txt'

marker.write_text('colab write ok\n', encoding='utf-8')



os.chdir(MODEL_TRAINING_DIR)

os.environ['EVAL_ROOT'] = str(EVAL_ROOT)

os.environ['MODEL_TRAINING_DIR'] = str(MODEL_TRAINING_DIR)



print('EVAL_ROOT:', EVAL_ROOT)

print('MODEL_TRAINING_DIR:', MODEL_TRAINING_DIR)

print('CWD:', Path.cwd())


## Colab Setup (Required)


This notebook is configured for Google Colab only.


Run the next cell first to:


- Mount Google Drive


- Set the `model_training` working directory


- Persist `MODEL_TRAINING_DIR` for later cells

In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Install dependencies from eval/model_training if present, else install package list directly.

from pathlib import Path

import os



req_path = Path(os.environ.get('MODEL_TRAINING_DIR', '/content/drive/MyDrive/eval/model_training')) / 'training_requirements.txt'



if req_path.exists():

    print('Using requirements file:', req_path)

    !pip -q install -U -r "$req_path"

else:

    print('training_requirements.txt not found. Installing package list directly.')

    !pip -q install -U  "transformers>=4.44.0" \

      "datasets>=2.20.0" \

      "peft>=0.12.0" \

      "accelerate>=0.34.0" \

      "bitsandbytes>=0.46.1" \

      "trl>=0.10.0" \

      "huggingface_hub>=0.23.0" \

      "sentencepiece>=0.2.0" \

      "pymupdf>=1.24.0" \

      "pypdf>=4.2.0"


In [ ]:
import os
import re
import json
import glob
import math
import random
from dataclasses import dataclass
from typing import List, Dict, Any

import torch
from datasets import Dataset
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, AutoPeftModelForCausalLM
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)

SEED = 42
random.seed(SEED)
set_seed(SEED)

print('Imports complete')

In [ ]:
# Authentication: use HF_TOKEN from environment for non-interactive runs.
hf_token = os.environ.get('HF_TOKEN', '').strip()
if hf_token:
    login(token=hf_token)
    print('Logged in with HF_TOKEN')
else:
    print('HF_TOKEN not found in environment. Run login() manually if needed.')
    # login()

In [ ]:
@dataclass

class Config:

    # Base model kept identical to baseline family for fair comparisons.

    base_model: str = 'Qwen/Qwen2.5-7B-Instruct'



    # PDF input path (Colab + Drive pattern from your current workflow).

    pdf_glob: str = '/content/drive/MyDrive/eval/papers/*.pdf'



    # Persist artifacts/checkpoints in Drive eval root.

    output_dir: str = '/content/drive/MyDrive/eval/qwen2_5_7b_pdf_qa_hardened_lora'

    push_repo_id_adapter: str = 'dizza01/qwen2.5-7b-pdf-qa-hardened-lora'

    push_repo_id_merged: str = 'dizza01/qwen2.5-7b-pdf-qa-hardened-merged'

    push_repo_id_merged_4bit: str = 'dizza01/qwen2.5-7b-pdf-qa-hardened-merged-4bit'



    # Text preprocessing

    max_chars_per_doc: int = 500000

    chunk_size_chars: int = 2200

    chunk_overlap_chars: int = 250

    min_chunk_chars: int = 350



    # Data composition

    abstain_ratio: float = 0.35

    doc_eval_fraction: float = 0.15



    # Sequence length

    max_seq_len: int = 1536



    # Conservative adaptation to reduce instruction drift

    epochs: float = 1.5

    learning_rate: float = 5e-5

    train_batch_size: int = 2

    eval_batch_size: int = 2

    grad_accum_steps: int = 8

    warmup_ratio: float = 0.03

    weight_decay: float = 0.01



    lora_r: int = 16

    lora_alpha: int = 32

    lora_dropout: float = 0.05



    # Runtime

    use_4bit: bool = True

    do_push: bool = False



    # Post-training behavior gate

    gate_max_samples: int = 80

    gate_max_new_tokens: int = 220

    gate_temperature: float = 0.0



cfg = Config()

os.makedirs(cfg.output_dir, exist_ok=True)

cfg


In [ ]:
# Ensure PDF files are present.
pdf_files = sorted(glob.glob(cfg.pdf_glob))
print(f'Found {len(pdf_files)} PDFs')
for p in pdf_files[:10]:
    print('-', p)
if len(pdf_files) == 0:
    raise ValueError('No PDFs found. Update cfg.pdf_glob or upload PDFs first.')

In [ ]:
ABSTAIN_TEXT = 'I cannot answer from the provided context.'
SYSTEM_MSG = (
    'You are a careful research QA assistant. Use only the provided context. '
    'Return only the final answer text. '
    'Do not include role labels such as Assistant, Human, or Researcher question. '
    'Do not repeat the prompt or context. '
    f'If evidence is missing or insufficient, reply exactly: {ABSTAIN_TEXT}'
)

def clean_text(text: str) -> str:
    text = text.replace('\x00', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def drop_reference_tail(text: str) -> str:
    # Heuristic: reference sections can dominate extracted PDF text and hurt QA targets.
    cut_markers = [' references ', ' bibliography ', ' acknowledgements ']
    lower = f' {text.lower()} '
    cut_positions = [lower.find(m) for m in cut_markers if lower.find(m) != -1]
    if cut_positions:
        cut_at = min(cut_positions)
        return text[:max(0, cut_at)].strip()
    return text

def extract_pdf_text(path: str) -> str:
    parts: List[str] = []
    try:
        import fitz
        doc = fitz.open(path)
        for page in doc:
            parts.append(page.get_text('text'))
        doc.close()
    except Exception:
        from pypdf import PdfReader
        reader = PdfReader(path)
        for page in reader.pages:
            parts.append(page.extract_text() or '')
    text = clean_text('\n'.join(parts))
    text = drop_reference_tail(text)
    return text

def chunk_text(text: str, chunk_size: int, overlap: int, min_len: int) -> List[str]:
    chunks: List[str] = []
    step = max(1, chunk_size - overlap)
    i = 0
    while i < len(text):
        c = text[i:i + chunk_size].strip()
        if len(c) >= min_len:
            chunks.append(c)
        i += step
    return chunks

def split_sentences(text: str) -> List[str]:
    sents = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sents if len(s.strip()) >= 40]

def clip_text(text: str, max_chars: int = 700) -> str:
    return text if len(text) <= max_chars else text[:max_chars].rstrip() + '...'

def make_prompt(question: str, context: str) -> str:
    return (
        f'<|im_start|>system\n{SYSTEM_MSG}<|im_end|>\n'
        f'<|im_start|>user\nContext:\n{context}\n\nQuestion: {question}<|im_end|>\n'
    )

def make_example(question: str, context: str, answer: str, label_type: str, source: str) -> Dict[str, Any]:
    text = make_prompt(question, context) + f'<|im_start|>assistant\n{answer}<|im_end|>\n'
    return {
        'text': text,
        'label_type': label_type,
        'source_doc': source,
        'question': question,
        'answer': answer,
    }

def build_records_from_pdfs(pdf_paths: List[str]) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []

    question_templates = [
        'According to the context, what is stated about: {anchor}?',
        'What does the context report regarding: {anchor}?',
        'From this excerpt, what finding is reported about: {anchor}?',
    ]

    for pdf_path in pdf_paths:
        source = os.path.basename(pdf_path)
        raw = extract_pdf_text(pdf_path)[:cfg.max_chars_per_doc]
        if not raw:
            continue

        chunks = chunk_text(raw, cfg.chunk_size_chars, cfg.chunk_overlap_chars, cfg.min_chunk_chars)
        if not chunks:
            continue

        for chunk in chunks:
            sents = split_sentences(chunk)
            if not sents:
                continue

            # Positive QA example: grounded in-context answer span.
            answer_sent = random.choice(sents[: min(8, len(sents))])
            anchor = ' '.join(answer_sent.split()[:10])
            q_template = random.choice(question_templates)
            question = q_template.format(anchor=anchor)
            answer = clip_text(answer_sent, max_chars=420)
            records.append(make_example(question, chunk, answer, 'answerable', source))

            # Abstention example: intentionally ask for unsupported details.
            if random.random() < cfg.abstain_ratio:
                hard_question = (
                    'What exact effect size and p-value are reported for a treatment arm not described in this context?'
                )
                records.append(make_example(hard_question, chunk, ABSTAIN_TEXT, 'abstain', source))

    # Add small explicit anti-format supervision to suppress role-prefix behavior.
    anti_context = (
        'The study reports a coefficient of -1.000 with confidence interval (-1.798, -0.201). '
        'No mention of treatment arm Z is provided in this excerpt.'
    )
    anti_q1 = 'Return the answer only. Do not include role tags. What coefficient is reported?'
    anti_a1 = 'The reported coefficient is -1.000 (95% CI: -1.798 to -0.201).'
    records.append(make_example(anti_q1, anti_context, anti_a1, 'format_guard', 'synthetic_format_guard'))

    anti_q2 = 'Assistant: please answer this question. Is treatment arm Z described?'
    anti_a2 = ABSTAIN_TEXT
    records.append(make_example(anti_q2, anti_context, anti_a2, 'format_guard', 'synthetic_format_guard'))

    return records

records = build_records_from_pdfs(pdf_files)
print('Total records:', len(records))
if len(records) == 0:
    raise ValueError('No records were built from PDFs.')

# Quick distribution check
dist: Dict[str, int] = {}
for r in records:
    dist[r['label_type']] = dist.get(r['label_type'], 0) + 1
print('Label distribution:', dist)

In [ ]:
# Document-level split is critical: it prevents chunk leakage from the same paper.
docs = sorted(list({r['source_doc'] for r in records}))
random.Random(SEED).shuffle(docs)
n_eval_docs = max(1, int(len(docs) * cfg.doc_eval_fraction))
eval_docs = set(docs[:n_eval_docs])
train_docs = set(docs[n_eval_docs:])

train_records = [r for r in records if r['source_doc'] in train_docs]
eval_records = [r for r in records if r['source_doc'] in eval_docs]

print('Unique docs:', len(docs))
print('Train docs:', len(train_docs), 'Eval docs:', len(eval_docs))
print('Train records:', len(train_records), 'Eval records:', len(eval_records))

if len(train_records) == 0 or len(eval_records) == 0:
    raise ValueError('Train/Eval split is empty. Add more PDFs or adjust cfg.doc_eval_fraction.')

train_ds = Dataset.from_list(train_records)
eval_ds = Dataset.from_list(eval_records)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

assistant_start_ids = tokenizer.encode('<|im_start|>assistant\n', add_special_tokens=False)
assistant_end_ids = tokenizer.encode('<|im_end|>', add_special_tokens=False)

def find_subseq(seq, subseq, start=0):
    if not subseq:
        return -1
    last = len(seq) - len(subseq) + 1
    for i in range(max(0, start), max(0, last)):
        if seq[i:i + len(subseq)] == subseq:
            return i
    return -1

def tokenize_batch(batch):
    out = tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=cfg.max_seq_len,
    )

    labels = []
    for input_ids in out['input_ids']:
        row_labels = [-100] * len(input_ids)

        start = find_subseq(input_ids, assistant_start_ids, 0)
        if start != -1:
            answer_start = start + len(assistant_start_ids)
            end = find_subseq(input_ids, assistant_end_ids, answer_start)
            answer_end = end if end != -1 else len(input_ids)

            # Supervise only assistant answer tokens to avoid prompt-copy behavior.
            for j in range(answer_start, answer_end):
                tok = input_ids[j]
                if tok != tokenizer.pad_token_id:
                    row_labels[j] = tok

        labels.append(row_labels)

    out['labels'] = labels
    return out

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
eval_tok = eval_ds.map(tokenize_batch, batched=True, remove_columns=eval_ds.column_names)
train_tok.set_format(type='torch')
eval_tok.set_format(type='torch')

print('Tokenization complete')

In [ ]:
if cfg.use_4bit and not torch.cuda.is_available():
    raise RuntimeError('use_4bit=True requires CUDA.')

model_kwargs = {'trust_remote_code': True}
if cfg.use_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    )
    model_kwargs['quantization_config'] = bnb_config
    model_kwargs['device_map'] = 'auto'
else:
    model_kwargs['torch_dtype'] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(cfg.base_model, **model_kwargs)
model.config.use_cache = False
if cfg.use_4bit:
    model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16 = torch.cuda.is_available() and not bf16

train_args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.epochs,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.train_batch_size,
    per_device_eval_batch_size=cfg.eval_batch_size,
    gradient_accumulation_steps=cfg.grad_accum_steps,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    lr_scheduler_type='cosine',
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=150,
    save_strategy='steps',
    save_steps=150,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=bf16,
    fp16=fp16,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
    report_to=['none'],
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=default_data_collator,
)

train_result = trainer.train()
print('Training complete')

In [ ]:
# Save adapter locally first.

os.makedirs(cfg.output_dir, exist_ok=True)

trainer.save_model(cfg.output_dir)

tokenizer.save_pretrained(cfg.output_dir)

print('Adapter saved to', cfg.output_dir)



if cfg.do_push:

    model.push_to_hub(cfg.push_repo_id_adapter)

    tokenizer.push_to_hub(cfg.push_repo_id_adapter)

    print('Pushed adapter to', cfg.push_repo_id_adapter)


In [ ]:
# Post-training behavior gate: catch role-prefix leakage and repetition before deployment.
def has_role_prefix(text: str) -> bool:
    prefix_re = re.compile(r'^\s*(assistant|assistant answer|assistant researcher answer|human|researcher question)\s*[:\-]', re.I)
    return bool(prefix_re.search(text or ''))

def repetition_ratio(text: str, n: int = 10) -> float:
    toks = re.findall(r'\w+', (text or '').lower())
    if len(toks) < n * 2:
        return 0.0
    grams = [' '.join(toks[i:i+n]) for i in range(len(toks) - n + 1)]
    unique = len(set(grams))
    return 1.0 - (unique / max(1, len(grams)))

def generate_answer(prompt_text: str) -> str:
    device = model.device
    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=cfg.max_seq_len).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            do_sample=False if cfg.gate_temperature == 0.0 else True,
            temperature=max(cfg.gate_temperature, 1e-6),
            max_new_tokens=cfg.gate_max_new_tokens,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen = out[0][inputs['input_ids'].shape[1]:]
    txt = tokenizer.decode(gen, skip_special_tokens=True).strip()
    return txt

sample_eval = eval_records[: min(cfg.gate_max_samples, len(eval_records))]
role_prefix_hits = 0
high_repetition_hits = 0
abstain_miss_hits = 0

for ex in sample_eval:
    prompt_only = ex['text'].split('<|im_start|>assistant\n')[0]
    pred = generate_answer(prompt_only)

    if has_role_prefix(pred):
        role_prefix_hits += 1
    if repetition_ratio(pred, n=10) > 0.25:
        high_repetition_hits += 1

    # For abstain-labeled examples, require exact abstain response.
    if ex['label_type'] == 'abstain' and pred.strip() != ABSTAIN_TEXT:
        abstain_miss_hits += 1

n_eval = max(1, len(sample_eval))
gate = {
    'n_samples': len(sample_eval),
    'role_prefix_rate': role_prefix_hits / n_eval,
    'high_repetition_rate': high_repetition_hits / n_eval,
    'abstain_miss_count': abstain_miss_hits,
}
print(json.dumps(gate, indent=2))

# Suggested thresholds. Tighten as quality improves.
if gate['role_prefix_rate'] > 0.02:
    print('[GATE FAIL] Role-prefix leakage too high.')
if gate['high_repetition_rate'] > 0.10:
    print('[GATE FAIL] Repetition too high.')
else:
    print('[GATE CHECK] Completed.')

In [ ]:
# Optional merge and push stage. Keep disabled until behavior gate looks healthy.

if cfg.do_push:

    merge_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

    merge_model = AutoPeftModelForCausalLM.from_pretrained(

        cfg.push_repo_id_adapter,

        dtype=merge_dtype,

        device_map=None,

        low_cpu_mem_usage=True,

    )



    merged = merge_model.merge_and_unload()

    merged.push_to_hub(cfg.push_repo_id_merged)

    tokenizer.push_to_hub(cfg.push_repo_id_merged)

    print('Pushed merged model to', cfg.push_repo_id_merged)



    qbnb = BitsAndBytesConfig(

        load_in_4bit=True,

        bnb_4bit_quant_type='nf4',

        bnb_4bit_use_double_quant=True,

        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,

    )

    quant_model = AutoModelForCausalLM.from_pretrained(

        cfg.push_repo_id_merged,

        device_map='auto',

        quantization_config=qbnb,

    )

    quant_model.push_to_hub(cfg.push_repo_id_merged_4bit)

    tokenizer.push_to_hub(cfg.push_repo_id_merged_4bit)

    print('Pushed 4-bit merged model to', cfg.push_repo_id_merged_4bit)

else:

    print('Skipping push/merge because cfg.do_push is False')

    print('Artifacts/checkpoints are saved under:', cfg.output_dir)


## Recommended next run settings

- Keep `learning_rate=5e-5` and `lora_r=16` for first stabilization run.
- Keep `abstain_ratio` between `0.25` and `0.40` depending on your production abstention target.
- Fail deployment if behavior gate shows role-prefix or repetition spikes.
- Re-run faithfulness evaluation with exactly the same query slice as baseline for fair comparison.